# Mathématiques appliquées à l'Intelligence Artificielle 
## 📝 Jour 4 : Réseaux de neurones
## 📚 Complément du cours 

2e année Bachelor Informatique

---

> © 2025 Mohamed EL AFRIT
> Ce contenu est distribué sous licence [Creative Commons BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.fr)
> [www.mohamedelafrit.com](https://www.mohamedelafrit.com)


## 🎯 Objectifs du Jour 4
- Comprendre le **neurone artificiel** et les **fonctions d'activation** (sigmoïde, ReLU)
- Construire un **réseau multicouche (MLP)** et comprendre le **forward pass**
- Saisir l'intuition de la **backpropagation**
- **Résoudre XOR** avec un réseau de neurones !


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DATASET FIL ROUGE — 30 développeurs
# ============================================================
# Colonnes : experience, nb_langages, niveau_etudes,
#            taille_entreprise, remote, salaire
noms_colonnes = ["experience", "nb_langages", "niveau_etudes",
                 "taille_entreprise", "remote", "salaire"]

data = np.array([
    [ 0, 2, 3,  150,  80, 28.0],  # 0  junior
    [ 1, 1, 2,   50, 100, 26.5],  # 1  autodidacte
    [ 1, 3, 4,  800,  40, 35.0],  # 2  sortie ecole
    [ 2, 2, 3,  200,  60, 32.0],  # 3
    [ 2, 4, 4, 3000,  20, 38.0],  # 4
    [ 0, 1, 1,   30,  50, 25.0],  # 5  debutant
    [ 3, 3, 3,  100,  80, 34.5],  # 6
    [ 1, 5, 4,   15, 100, 42.0],  # 7  junior polyglotte
    [ 4, 3, 3,  500,  60, 38.5],  # 8
    [ 5, 4, 4, 1200,  40, 44.0],  # 9
    [ 5, 3, 3,  300,  80, 41.0],  # 10
    [ 6, 5, 4, 2000,  30, 48.0],  # 11
    [ 6, 4, 3,  150,  90, 43.5],  # 12
    [ 7, 5, 4,  800,  50, 50.0],  # 13
    [ 7, 3, 2,   80,  70, 39.0],  # 14 exp mais Bac+2
    [ 4, 6, 5,  400, 100, 47.0],  # 15
    [ 8, 4, 4, 1500,  40, 51.0],  # 16
    [ 5, 2, 3, 4000,  10, 40.0],  # 17
    [ 9, 5, 4,  600,  60, 52.0],  # 18
    [10, 6, 4, 2500,  30, 55.0],  # 19
    [10, 4, 3,  200,  80, 48.5],  # 20
    [11, 5, 5, 1000,  50, 58.0],  # 21
    [12, 7, 4, 3500,  20, 60.0],  # 22
    [ 9, 3, 3,   60,  90, 42.0],  # 23 senior sous-paye
    [13, 6, 4,  800,  70, 62.0],  # 24
    [11, 4, 4, 5000,  10, 54.0],  # 25
    [15, 8, 4, 1200,  50, 65.0],  # 26
    [17, 6, 5, 2000,  40, 68.0],  # 27
    [20, 7, 4,  500,  60, 72.0],  # 28
    [18, 5, 3,  100,  80, 58.0],  # 29
])

experience = data[:, 0]
nb_langages = data[:, 1]
salaire = data[:, 5]
salaire_eleve = (salaire > 45).astype(int)
n = len(data)

print(f"Dataset charge : {n} developpeurs, {data.shape[1]} colonnes")
print(f"Classe 0 (<=45k) : {np.sum(salaire_eleve==0)}, Classe 1 (>45k) : {np.sum(salaire_eleve==1)}")


---
## 1. Fonctions d'activation

| Fonction | Formule | Sortie |
|----------|---------|--------|
| Sigmoïde | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $(0, 1)$ |
| ReLU | $\max(0, z)$ | $[0, +\infty)$ |
| Signe | $\{0, 1\}$ | perceptron J3 |


In [ ]:
# Tracer sigmoïde et ReLU
z = np.linspace(-6, 6, 200)
sigmoid = 1 / (1 + np.exp(-z))
relu = np.maximum(0, z)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(z, sigmoid, 'b-', lw=2.5); ax1.axhline(0.5, color='gray', ls=':')
ax1.set_title("Sigmoïde"); ax1.grid(True, alpha=0.3)
ax2.plot(z, relu, 'r-', lw=2.5); ax2.set_title("ReLU"); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## 2. Forward pass d'un réseau 2-2-1

> $$\mathbf{h} = \sigma(\mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)})$$
> $$\hat{y} = \sigma(\mathbf{w}^{(2)T}\mathbf{h} + b^{(2)})$$


In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

x = np.array([5.0, 4.0])  # un dev
W1 = np.array([[0.3, 0.5], [-0.2, 0.4]]); b1 = np.array([-2.0, 0.5])
w2 = np.array([0.8, 0.6]); b2 = -0.5

z1 = W1 @ x + b1; h = sigmoid(z1)
z2 = w2 @ h + b2; y_pred = sigmoid(z2)

print(f"Couche cachée : z1={z1}, h={np.round(h,4)}")
print(f"Sortie : z2={z2:.4f}, ŷ={y_pred:.4f}")
print(f"Prédiction : {y_pred:.1%} → classe {1 if y_pred>0.5 else 0}")


---
## 3. Backpropagation — Résoudre XOR !


In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))
def sigmoid_d(a): return a*(1-a)

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)

np.random.seed(42)
W1 = np.random.randn(2,4)*0.5; b1 = np.zeros((1,4))
W2 = np.random.randn(4,1)*0.5; b2 = np.zeros((1,1))

for epoch in range(5000):
    h = sigmoid(X@W1+b1); out = sigmoid(h@W2+b2)
    loss = np.mean((y-out)**2)
    d_out = (out-y)*sigmoid_d(out)
    W2 -= 1.0*(h.T@d_out)/4; b2 -= 1.0*np.mean(d_out,axis=0,keepdims=True)
    d_h = (d_out@W2.T)*sigmoid_d(h)
    W1 -= 1.0*(X.T@d_h)/4; b1 -= 1.0*np.mean(d_h,axis=0,keepdims=True)
    if epoch%1000==999:
        acc=np.mean((out>0.5).flatten()==y.flatten())
        print(f"Epoch {epoch+1} : loss={loss:.4f}, acc={acc:.0%}")

print(f"\nPrédictions : {out.flatten().round(3)}")
print(f"Attendu     : {y.flatten().astype(int)}")
print("\n🎉 Le réseau résout XOR !")


---
## 4. Application au dataset salaire


In [ ]:
def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
def sigmoid_d(a): return a*(1-a)

X_raw = data[:,:2]; y_cls = salaire_eleve.reshape(-1,1).astype(float)
X_n = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)  # normalisation !

np.random.seed(42)
W1=np.random.randn(2,4)*0.5; b1=np.zeros((1,4))
W2=np.random.randn(4,1)*0.5; b2=np.zeros((1,1))
hist_loss=[]

for ep in range(2000):
    h=sigmoid(X_n@W1+b1); out=sigmoid(h@W2+b2)
    loss=np.mean((y_cls-out)**2); hist_loss.append(loss)
    do=(out-y_cls)*sigmoid_d(out)
    W2-=0.5*(h.T@do)/n; b2-=0.5*np.mean(do,axis=0,keepdims=True)
    dh=(do@W2.T)*sigmoid_d(h)
    W1-=0.5*(X_n.T@dh)/n; b1-=0.5*np.mean(dh,axis=0,keepdims=True)

acc=np.mean((out>0.5)==y_cls)
print(f"Accuracy réseau 2-4-1 : {acc:.0%}")

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
ax1.plot(hist_loss,'b-',lw=1); ax1.set_title("Convergence"); ax1.grid(True,alpha=0.3)

xx,yy=np.meshgrid(np.linspace(X_n[:,0].min()-1,X_n[:,0].max()+1,100),np.linspace(X_n[:,1].min()-1,X_n[:,1].max()+1,100))
grid=np.c_[xx.ravel(),yy.ravel()]; hg=sigmoid(grid@W1+b1); zg=sigmoid(hg@W2+b2).reshape(xx.shape)
ax2.contourf(xx,yy,zg,levels=20,cmap='RdYlGn',alpha=0.5)
ax2.contour(xx,yy,zg,levels=[0.5],colors='black',linewidths=2)
for c,col in [(0,'#e74c3c'),(1,'#2ecc71')]:
    m=y_cls.flatten()==c; ax2.scatter(X_n[m,0],X_n[m,1],c=col,s=60,edgecolors='white')
ax2.set_title(f"Surface de décision ({acc:.0%})"); plt.tight_layout(); plt.show()


---
## ✅ Bilan des 4 jours

| Jour | Thème | Formule clé |
|------|-------|-------------|
| 1 | Représentation | $\hat{y} = \mathbf{w}^T\mathbf{x} + b$ |
| 2 | Optimisation | $w \leftarrow w - \alpha \nabla J$ |
| 3 | Classification | $\hat{y} = \text{signe}(\mathbf{w}^T\mathbf{x}+b)$ |
| 4 | Réseaux | $\mathbf{h} = \sigma(\mathbf{W}^{(1)}\mathbf{x}+\mathbf{b}^{(1)})$ |

> 🎓 Vous avez les **fondations mathématiques** pour comprendre l'IA moderne !
